### 分子のグラフ表現

#### SMILES (simplified molecular input line entry system)

分子グラフをテキストデータに変換する方法の1つ

昔はde facto standardであったもの

文法に従わないものや原子価の制約を満たさないとerror

文法に従っていても対応する分子構造に誤りがある場合もerror

ニューラルネットワークで100%制約を満たせるSMILEを出力をするのが非常に困難

In [1]:
# 化合物データを取り扱うrdkitライブラリをインストール

!pip install rdkit

  Obtaining dependency information for rdkit from https://files.pythonhosted.org/packages/94/85/d8d7da7ba8d00ad8977193f30d0062324aa9279e8af67e413674d3f60483/rdkit-2024.9.6-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/22.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/22.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/22.5 MB 217.9 kB/s eta 0:01:44
   ---------------------------------------- 0.1/22.5 MB 435.7 kB/s eta 0:00:52
    --------------------------------------- 0.3/22.5 MB 2.0 MB/s eta 0:00:12
   - -------------------------------------- 0.8/22.5 MB 4.3 MB/s eta 0:00:06
   -- ------------------------------------- 1.4/22.5 MB 5.6 MB/s eta 0:00:04
   --- ------------------------------------ 2.1/22.5 MB 7.5 MB/s eta 0:00:03
   ----- ---------------------------------- 3.1/22.5 MB 8.9 MB/s eta 0:00:03
   ------- -------------------------------- 4.1/22.5 MB 11.0 MB/s eta 0:00:02
   --------- -------------------

In [2]:
# SMILESで表された文字列を分子グラフにしてみる
# ちなみに分子はカフェイン

from rdkit import Chem
from rdkit.Chem import Draw

mol = Chem.MolFromSmiles('CN1C=NC2=C1C(=O)N(C(=O)N2C)C')
Draw.MolToFile(mol, 'caffeine.svg')

#### SELFIES (self-referencing embedded strings)

SMILESの弱点を克服した記述法

任意のSELFIES系列を必ず正しい分子グラフに変換可能

すべての分子を表現可能

文脈自由文法をベースにして作成

In [3]:
# SELFIESを扱うライブラリをインストール

!pip install selfies

  Obtaining dependency information for selfies from https://files.pythonhosted.org/packages/d8/28/e55116335f778c734b1211d3abacd429599a2f0f664d267eb1282f9906a2/selfies-2.2.0-py3-none-any.whl.metadata


In [4]:
# selfiesのテスト

import selfies as sf

print(sf.__version__)
print(sf.decoder('[F][=C][=C][#N]'))
print(sf.decoder('[C][Branch1][C][F][Cl]'))
print(sf.decoder('[C][C][C][=Ring1][Ring1][=C]'))

2.1.1
FC=C=N
C(F)Cl
C=1CC=1C


#### 分子記述子

分子を数値やベクトルで表す方法

分子量，環の数，オクタノール/水分配係数 (octanol-water partition coefficient) etc. がある

#### オクタノール/水分配係数 ($\log P$)

オクタノールと水を混ぜた液体を溶媒にして

対象物質それぞれの溶媒中における濃度比の対数を取ったもの

この値が大きいほど親水性/逆だと疎水性

In [6]:
# カフェインのlogPを計算

from rdkit.Chem import Descriptors

mol = Chem.MolFromSmiles('CN1C=NC2=C1C(=O)N(C(=O)N2C)C')
logp = Descriptors.MolLogP(mol)
print('logp = {0:.3f}'.format(logp))

logp = -1.029


#### ECFP (extended-connectivity fingerprint)

分子を特徴付けるようなベクトル

2値ベクトルであることが多い

各原子に対して識別子と呼ばれる整数値を割り当て

原子の周りの情報や隣接する識別子を用いて繰り返し更新

In [7]:
# カフェインのECFP計算

import torch
from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect

mol = Chem.MolFromSmiles('CN1C=NC2=C1C(=O)N(C(=O)N2C)C')
fp = GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2**11)
fp_tensor = torch.tensor(fp)
fp_idx_tensor = torch.tensor(fp.GetOnBits())
print('fp_tensor = {}'.format(fp_tensor))
print('shape = {}'.format(fp_tensor.shape))
print('non-zero indices: {}'.format(fp_idx_tensor))

[02:39:48] DEPRECATION WARNING: please use MorganGenerator


fp_tensor = tensor([0, 0, 0,  ..., 0, 0, 0])
shape = torch.Size([2048])
non-zero indices: tensor([  33,  314,  378,  400,  463,  504,  564,  650,  771,  932,  935, 1024,
        1057, 1145, 1203, 1258, 1307, 1354, 1380, 1409, 1440, 1452, 1517, 1696,
        1873])


#### Tanimoto similarity

2値ベクトル同士の類似度

${\bf x}$，${\bf y}$の類似度は以下のように定義される

$\frac {{\bf x}^\top {\bf y}} {{\bf 1}^\top {\bf x} + {\bf 1}^\top {\bf y} - {\bf x}^\top {\bf y}}$

In [8]:
# カフェインとテオフィリンのTanimoto similarity

from rdkit import DataStructs

caffeine = Chem.MolFromSmiles('CN1C=NC2=C1C(=O)N(C(=O)N2C)C')
theophylline = Chem.MolFromSmiles('Cn1c2c(c(=O)n(c1=O)C)[nH]cn2')
fp_c = GetMorganFingerprintAsBitVect(caffeine, radius=2, nBits=2**11)
fp_t = GetMorganFingerprintAsBitVect(theophylline, radius=2, nBits=2**11)
print('Tanimoto similarity: {0:.3f}'.format(DataStructs.FingerprintSimilarity(fp_c, fp_t)))

Tanimoto similarity: 0.457


[02:45:01] DEPRECATION WARNING: please use MorganGenerator
[02:45:01] DEPRECATION WARNING: please use MorganGenerator
